# PYNQ-Z2 四通道逻辑分析仪 · Notebook v2

本 Notebook 按 7 个步骤完成一次完整采集，每一步都会打印带编号的状态信息，便于演示和排错。

| 项目 | 取值 |
|---|---|
| 采样率 | 1 MSa/s（由 RTL 固定，50 MHz ÷ 50） |
| 缓冲深度 | 2048 点 |
| 预触发 / 后触发 | 256 / 256 点 |
| 触发模式 | 上升沿 / 下降沿 / 任意边沿 |
| 激励源 | 内部测试模式（TEST_MODE=1） |

**运行前提**：`logic_analyzer.bit` 与 `logic_analyzer.hwh` 必须与本 Notebook 位于同一目录且基名相同。


## 1. 加载 Overlay


In [ ]:
from pynq import Overlay
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os

STEPS = []
BITFILE = "logic_analyzer.bit"
HWHFILE = "logic_analyzer.hwh"

print("[1/7] 加载 Overlay ...")
for f in (BITFILE, HWHFILE):
    if not os.path.exists(f):
        raise FileNotFoundError(f"缺少 {f}，请与 Notebook 放在同一目录")
print(f"      找到 {BITFILE} ({os.path.getsize(BITFILE)/1024/1024:.2f} MB)")

ol = Overlay(BITFILE)
print(f"      IP 列表: {list(ol.ip_dict.keys())}")

gpio_ctrl = ol.axi_gpio_0.channel1
gpio_status = ol.axi_gpio_1.channel1
bram = ol.axi_bram_ctrl_0

def write_ctrl(value):
    """兼容不同 PYNQ 版本的 AXI GPIO 写接口。"""
    try:
        gpio_ctrl.write(value, 0xFFFFFFFF)
    except TypeError:
        gpio_ctrl.write(value)


STEPS.append("[1/7] Overlay 加载完成")
print("[OK] Overlay 加载完成，控制/状态/BRAM 句柄就绪")


## 2. 配置采集参数


In [ ]:
DEPTH = 2048
PRE_TRIGGER = 256
POST_TRIGGER = 256
TRIGGER_CH = 0
TRIGGER_MODE = 0
TEST_MODE = 1
TIMEOUT_S = 5.0

MODE_NAME = {0: "上升沿", 1: "下降沿", 2: "任意边沿"}
SOURCE_NAME = {1: "内部测试模式", 0: "Pmod 外部输入"}

ctrl_idle = (
    ((PRE_TRIGGER & 0x3FF) << 16)
    | ((POST_TRIGGER & 0x3FF) << 6)
    | ((TRIGGER_MODE & 0x3) << 4)
    | ((TRIGGER_CH & 0x3) << 2)
    | ((TEST_MODE & 0x1) << 31)
)

print("[2/7] 采集参数")
print("      采样率     : 1 MSa/s（RTL 固定，50 MHz ÷ 50）")
print(f"      缓冲深度   : {DEPTH} 点")
print(f"      预/后触发  : {PRE_TRIGGER} / {POST_TRIGGER} 点")
print(f"      触发通道   : CH{TRIGGER_CH}")
print(f"      触发模式   : {TRIGGER_MODE} - {MODE_NAME.get(TRIGGER_MODE, '保留')}")
print(f"      激励源     : {SOURCE_NAME[TEST_MODE]}")
print(f"      ctrl_idle  : 0x{ctrl_idle:08X}")

STEPS.append("[2/7] 参数配置完成")


## 3. 单次采集：clear → arm → 等待 done


In [ ]:
def capture_once():
    """清状态、arm、轮询 done，返回 (status, elapsed)。"""
    write_ctrl(ctrl_idle | 0x2); time.sleep(0.01)
    write_ctrl(ctrl_idle); time.sleep(0.01)
    write_ctrl(ctrl_idle | 0x1); time.sleep(0.01)
    write_ctrl(ctrl_idle)
    t0 = time.time()
    while True:
        status = gpio_status.read() & 0xFFFFFFFF
        if status & 0x1:
            return status, time.time() - t0
        if time.time() - t0 > TIMEOUT_S:
            raise TimeoutError(f"采集超时，status=0x{status:08X}")

print("[3/7] 单次采集 ...")
status, elapsed = capture_once()
done = status & 0x1
triggered = (status >> 1) & 0x1
trigger_ptr = (status >> 2) & 0x7FF
print(f"      done={done} triggered={triggered} trigger_ptr={trigger_ptr}")
print(f"      耗时 {elapsed*1000:.1f} ms")
if done != 1 or triggered != 1:
    raise RuntimeError("采集未完成或未触发，请检查 TEST_MODE 与触发参数")

STEPS.append(f"[3/7] 采集完成 trigger_ptr={trigger_ptr}")
print("[OK] 采集完成")


## 4. 读取 BRAM 并还原时间顺序


In [ ]:
print("[4/7] 读取 BRAM ...")
words = [bram.mmio.read(4 * i) for i in range(DEPTH)]
probes = np.array([w & 0xF for w in words], dtype=np.uint8)
timestamps = np.array([(w >> 4) & 0x0FFFFFFF for w in words], dtype=np.uint32)

start = (trigger_ptr - PRE_TRIGGER) % DEPTH
probes_ordered = np.roll(probes, -start)
timestamps_ordered = np.roll(timestamps, -start)

valid_len = PRE_TRIGGER + POST_TRIGGER + 1
probes_valid = probes_ordered[:valid_len]
timestamps_valid = timestamps_ordered[:valid_len]

ch0 = (probes_valid >> 0) & 1
ch1 = (probes_valid >> 1) & 1
ch2 = (probes_valid >> 2) & 1
ch3 = (probes_valid >> 3) & 1

print(f"      BRAM 读取 {DEPTH} 个样本")
print(f"      触发点地址 {trigger_ptr}，还原起点 {start}")
print(f"      valid samples: {valid_len}")
print(f"      first probes: {probes_valid[:20]}")
print(f"      时间戳范围: {timestamps_valid[0]} → {timestamps_valid[-1]}")

STEPS.append(f"[4/7] 读取完成 valid={valid_len}")
print("[OK] 数据读取与还原完成")


## 5. 绘制四通道波形


In [ ]:
print("[5/7] 绘制波形 ...")
fig, axes = plt.subplots(4, 1, figsize=(14, 7), sharex=True)
channels = [ch0, ch1, ch2, ch3]
for i, (ax, ch) in enumerate(zip(axes, channels)):
    ax.step(range(len(ch)), ch, where="post", linewidth=0.9)
    ax.axvline(PRE_TRIGGER, color="red", linestyle="--", alpha=0.8,
               label="Trigger" if i == 0 else None)
    ax.set_ylabel(f"CH{i}")
    ax.set_ylim(-0.2, 1.2)
    ax.grid(True)
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Sample index")
fig.suptitle("PYNQ-Z2 4-Channel Logic Analyzer")
fig.tight_layout()
fig.savefig("waveform_4ch.png", dpi=150)
plt.show()
print("      已保存 waveform_4ch.png")

STEPS.append("[5/7] 波形绘制完成")


## 6. 导出 CSV


In [ ]:
print("[6/7] 导出 CSV ...")
df = pd.DataFrame({
    "index": range(valid_len),
    "timestamp": timestamps_valid,
    "ch0": ch0,
    "ch1": ch1,
    "ch2": ch2,
    "ch3": ch3,
})
csv_name = "logic_capture.csv"
df.to_csv(csv_name, index=False)
print(f"      已保存 {csv_name}（{len(df)} 行 × {df.shape[1]} 列）")

write_ctrl(ctrl_idle | 0x2); time.sleep(0.01)
write_ctrl(ctrl_idle)

STEPS.append(f"[6/7] CSV 导出 {csv_name} ({len(df)} 行)")
print("[OK] CSV 导出完成，状态已清零")


## 7. 连续 10 次稳定性测试


In [ ]:
print("[7/7] 连续 10 次采集 ...")
results = []
for i in range(10):
    st, el = capture_once()
    d, tg, ptr = st & 1, (st >> 1) & 1, (st >> 2) & 0x7FF
    results.append((d, tg, ptr))
    print(f"      run {i+1:02d}: done={d} triggered={tg} trigger_ptr={ptr} {el*1000:.1f} ms")

passed = sum(1 for d, tg, _ in results if d == 1 and tg == 1)
print(f"      PASS {passed}/10")
STEPS.append(f"[7/7] 连续 10 次 {passed}/10 通过")
if passed != 10:
    raise RuntimeError("连续采集未全部通过")


## 运行状态汇总


In [ ]:
print("本次运行状态汇总")
print("-" * 44)
for s in STEPS:
    print(s)
print("-" * 44)
print("全部 7 个步骤完成")
